In [1]:
import pandas as pd

orders = pd.read_csv(r"C:\Users\matth\Downloads\Customer segementation Unzipped Data\orders.csv")

prior = pd.read_csv(r"C:\Users\matth\Downloads\Customer segementation Unzipped Data\order_products__prior.csv")

train = pd.read_csv(r"C:\Users\matth\Downloads\Customer segementation Unzipped Data\order_products__train.csv")

products = pd.read_csv(r"C:\Users\matth\Downloads\Customer segementation Unzipped Data\products.csv")

aisles = pd.read_csv(r"C:\Users\matth\Downloads\Customer segementation Unzipped Data\aisles.csv")

departments = pd.read_csv(r"C:\Users\matth\Downloads\Customer segementation Unzipped Data\departments.csv")

# Customer Segmentation: Louvain Community Detection

## Data Preparation

Before constructing the customer-aisle bipartite graph and applying Louvain community detection, the integrated analytical dataset (`preprocessed_df`) from `01_data_understanding_preprocessing_cohort_analysis.ipynb` was recreated.

The prior and train order-product datasets were combined and merged with customer order information, product information, aisle information, and department information. The resulting dataset serves as the basis for constructing the weighted customer-aisle bipartite graph.


In [2]:
# Combine prior and train order-product datasets
order_products = pd.concat([prior, train],axis=0,ignore_index=True)

# Merge with orders table
preprocessed_df = order_products.merge(orders,on="order_id",how="left")

# Merge with products table
preprocessed_df = preprocessed_df.merge(products,on="product_id",how="left")

# Merge with aisles table
preprocessed_df = preprocessed_df.merge(aisles,on="aisle_id",how="left")

# Merge with departments table
preprocessed_df = preprocessed_df.merge(departments,on="department_id",how="left")

print("Final preprocessed dataset:")
print(preprocessed_df.isnull().sum())
print("Duplicates:", preprocessed_df.duplicated().sum())
print(preprocessed_df.shape)
preprocessed_df.head()

Final preprocessed dataset:
order_id                        0
product_id                      0
add_to_cart_order               0
reordered                       0
user_id                         0
eval_set                        0
order_number                    0
order_dow                       0
order_hour_of_day               0
days_since_prior_order    2078068
product_name                    0
aisle_id                        0
department_id                   0
aisle                           0
department                      0
dtype: int64
Duplicates: 0
(33819106, 15)


,order_id,product_id,add_to_cart_order,reordered,user_id,eval_set,order_number,order_dow,order_hour_of_day,days_since_prior_order,product_name,aisle_id,department_id,aisle,department
0,2,33120,1,1,202279,prior,3,5,9,8.0,Organic Egg Whites,86,16,eggs,dairy eggs
1,2,28985,2,1,202279,prior,3,5,9,8.0,Michigan Organic Kale,83,4,fresh vegetables,produce
2,2,9327,3,0,202279,prior,3,5,9,8.0,Garlic Powder,104,13,spices seasonings,pantry
3,2,45918,4,1,202279,prior,3,5,9,8.0,Coconut Butter,19,13,oils vinegars,pantry
4,2,30035,5,0,202279,prior,3,5,9,8.0,Natural Sweetener,17,13,baking ingredients,pantry


In [3]:
# Create customer-aisle purchase counts as weighted edges
customer_aisle_edges = (preprocessed_df.groupby(["user_id", "aisle_id", "aisle"], as_index=False).size().rename(columns={"size": "weight"}))
print(customer_aisle_edges.shape)
customer_aisle_edges.head()

(5919840, 4)


,user_id,aisle_id,aisle,weight
0,1,21,packaged cheese,9
1,1,23,popcorn jerky,13
2,1,24,fresh fruits,5
3,1,45,candy chocolate,2
4,1,53,cream,3


In [4]:
import networkx as nx

B = nx.Graph()
# Define customer nodes using the original user IDs
customer_nodes = customer_aisle_edges["user_id"].unique()
# Define aisle nodes as negative aisle IDs to avoid overlap with customer IDs
aisle_nodes = -customer_aisle_edges["aisle_id"].unique()


# Build weighted customer-aisle bipartite graph
B.add_nodes_from(customer_nodes, bipartite="customer")
B.add_nodes_from(aisle_nodes, bipartite="aisle")

B.add_weighted_edges_from(
    (row.user_id, -row.aisle_id, row.weight)
    for row in customer_aisle_edges.itertuples(index=False)
)
print("Nodes:", B.number_of_nodes())
print("Edges:", B.number_of_edges())

Nodes: 206343
Edges: 5919840


In [5]:
# Evaluate different Louvain resolution values
resolution_values = [0.8, 0.9, 1.0, 1.1, 1.2]

resolution_results = []

for res in resolution_values:
    communities_res = nx.community.louvain_communities(B, weight="weight", resolution=res, seed=42)
    
    customer_counts = []
    aisle_counts = []
    
    for community in communities_res:
        num_customers = sum(1 for node in community if node > 0)
        num_aisles = sum(1 for node in community if node < 0)
        
        if num_customers > 0:
            customer_counts.append(num_customers)
            aisle_counts.append(num_aisles)
    
    resolution_results.append({
        "resolution": res,
        "num_communities": len(communities_res),
        "largest_segment": max(customer_counts),
        "smallest_segment": min(customer_counts),
        "median_segment_size": pd.Series(customer_counts).median(),
        "total_customers": sum(customer_counts),
        "total_aisles": sum(aisle_counts)
    })

resolution_results_df = pd.DataFrame(resolution_results)
resolution_results_df

,resolution,num_communities,largest_segment,smallest_segment,median_segment_size,total_customers,total_aisles
0,0.8,3,118095,8041,80073.0,206209,134
1,0.9,3,95273,24482,86454.0,206209,134
2,1.0,5,59523,27465,35699.0,206209,134
3,1.1,13,40445,1552,11348.0,206209,134
4,1.2,21,33351,280,7801.0,206209,134


In [6]:
# Run final Louvain community detection with resolution = 1.0
final_resolution = 1.0
communities = nx.community.louvain_communities(
    B,
    weight="weight",
    resolution=final_resolution,
    seed=42
)
print("Number of communities:", len(communities))

Number of communities: 5


In [7]:
# Assign Louvain community labels to customer nodes
customer_segments = []
for community_id, community in enumerate(communities):
    for node in community:
        if node > 0:  
            customer_segments.append({
                "user_id": node,
                "louvain_segment": community_id
            })

customer_segments_df = pd.DataFrame(customer_segments)
customer_segments_df.head()

,user_id,louvain_segment
0,2,0
1,65539,0
2,131076,0
3,196613,0
4,65540,0


In [8]:
print("Customers in original data:", preprocessed_df["user_id"].nunique())
print("Customers assigned to segments:", customer_segments_df["user_id"].nunique())

Customers in original data: 206209
Customers assigned to segments: 206209


In [9]:
# Check size of each Louvain community
customer_segments_df["louvain_segment"].value_counts().sort_index()

louvain_segment
0    27465
1    59523
2    30950
3    35699
4    52572
Name: count, dtype: int64

In [10]:
customer_segments_df.to_csv("louvain_customer_segments.csv", index=False)